<a href="https://colab.research.google.com/github/jakob-steuer/chemoinformatics_course/blob/main/chemoinformatics_7_qsar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QSAR — and how to find out where your model is lying to you

In the pharmacophore notebook *you* decided which features matter. That works when you
understand the target. **QSAR** takes the other route: hand a model a pile of molecules with
measured numbers and let it work out the relationship.

This notebook does that twice.

| | dataset | question | what it teaches |
|---|---|---|---|
| **Part 1** | blood–brain barrier, 2 000 drugs | does it cross? (yes/no) | why accuracy lies, what a baseline is for |
| **Part 2** | **your own target, from ChEMBL** | how strongly does it bind? (pIC₅₀) | selectivity, leakage, applicability domain |
| **Part 3** | both | where do my compounds sit? | chemical space, series, islands |

**Part 2 is your project work.** You edit four lines, and the figure that comes out is the kind of
thing that belongs in your final presentation. Everything before it exists so that you can read
that figure critically.

What you need from the pharmacophore notebook: your drug's name, your two proteins, and the
3–5 compounds you shortlisted in its Exercise 7.

> ⏳ **Start the next cell now** — the install takes about a minute, and you can read on while it
> runs. Cells marked ⏳ later on are the other slow ones; start each as soon as you reach it
> rather than after you have finished reading around it.

In [ ]:
# Install the packages Colab does not ship with (about a minute).
%pip install -q rdkit umap-learn

In [ ]:
import os
import time
import warnings
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay,
                             r2_score, mean_absolute_error)

warnings.filterwarnings("ignore")     # UMAP and sklearn are chatty about things we know
RDLogger.DisableLog("rdApp.*")        # RDKit complains loudly about every unparsable SMILES
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 11, "figure.dpi": 110})

morgan = GetMorganGenerator(radius=2, fpSize=2048)
print("ready.")

In [ ]:
# @title Plumbing — run once. PubChem name lookup, the ChEMBL fetch, two plot helpers. { display-mode: "form" }

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
CHEMBL_URL = f"{CHEMBL_BASE}/activity.json"
CHEMBL_FIELDS = "canonical_smiles,pchembl_value,molecule_chembl_id,standard_type"

# name -> (ChEMBL target id, mutation string or None for the wild-type protein)
TARGETS = {
    # --- two related proteins (selectivity track) -----------------------------------
    "AChE":          ("CHEMBL220",  None),   "BChE":        ("CHEMBL1914", None),
    "thrombin":      ("CHEMBL204",  None),   "factor Xa":   ("CHEMBL244",  None),
    "ER-alpha":      ("CHEMBL206",  None),   "ER-beta":     ("CHEMBL242",  None),
    "EGFR":          ("CHEMBL203",  None),   "HER2":        ("CHEMBL1824", None),
    "MAO-A":         ("CHEMBL1951", None),   "MAO-B":       ("CHEMBL2039", None),
    "CA II":         ("CHEMBL205",  None),   "CA IX":       ("CHEMBL3594", None),
    "COX-1":         ("CHEMBL221",  None),   "COX-2":       ("CHEMBL230",  None),
    "KIT":           ("CHEMBL1936", None),   "PDGFR-alpha": ("CHEMBL2007", None),
    "ABL1":          ("CHEMBL1862", None),   "VEGFR2":      ("CHEMBL279",  None),
    "ALK":           ("CHEMBL4247", None),   "BRAF":        ("CHEMBL5145", None),
    "HIV-1 protease": ("CHEMBL243", None),
    # --- the same protein with one residue changed (resistance track) ---------------
    "EGFR T790M":    ("CHEMBL203",  "L858R,T790M"),
    "ABL1 T315I":    ("CHEMBL1862", "T315I"),
    "ALK L1196M":    ("CHEMBL4247", "L1196M"),
    "BRAF V600E":    ("CHEMBL5145", "V600E"),
}

# targets whose clinical mutants exist in ChEMBL only as one-off patient panels
NO_USABLE_MUTANT = {
    "HIV-1 protease": "every resistant isolate carries a different combination of 5-15 "
                      "mutations, so there is no set of measurements large enough to train on.",
}


def smiles_from_pubchem(name):
    """A compound name (or a SMILES string) in, SMILES out. None if PubChem has no match."""
    if Chem.MolFromSmiles(name) is not None:
        return name
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           f"{requests.utils.quote(name)}/property/SMILES/TXT")
    try:
        answer = requests.get(url, timeout=30)
        if answer.ok and answer.text.strip():
            return answer.text.strip().split("\n")[0]
    except Exception as error:
        print(f"  PubChem lookup failed for {name!r}: {error}")
    return None


def _chembl(query, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            answer = requests.get(CHEMBL_URL, params=query, timeout=300)
            answer.raise_for_status()
            return answer.json()
        except Exception as error:
            print(f"    ChEMBL attempt {attempt}/{attempts} failed: {error}")
            if attempt == attempts:
                raise
            time.sleep(3 * attempt)


def find_target(query, organism="Homo sapiens"):
    """Look a protein up in ChEMBL by name. Use this when your target is not in TARGETS."""
    answer = requests.get(f"{CHEMBL_BASE}/target/search.json",
                          params={"q": query, "limit": 25,
                                  "only": "target_chembl_id,pref_name,organism,target_type"},
                          timeout=60)
    answer.raise_for_status()
    rows = [{"ChEMBL id": t["target_chembl_id"], "protein": t["pref_name"]}
            for t in answer.json()["targets"]
            if t["target_type"] == "SINGLE PROTEIN"
            and (organism is None or t["organism"] == organism)]
    if not rows:
        print(f"nothing matching {query!r} in {organism}. Try a different name, or pass "
              f"organism=None.")
    return pd.DataFrame(rows)


def find_mutations(target, pages=3, page=1000):
    """Which mutants have actually been measured against this target, and how often.

    ChEMBL stores a mutation on the *assay*, not as a separate protein, so this reads the
    activity records and counts the variant labels. It samples three pages spread across
    the target rather than reading everything, which is enough to find anything common.
    """
    query = {"target_chembl_id": target, "pchembl_value__isnull": "false",
             "assay_type": "B", "only": "assay_variant_mutation"}
    total = _chembl({**query, "limit": 1})["page_meta"]["total_count"]
    offsets = (np.linspace(0, max(total - page, 0), pages).astype(int) if total > page else [0])
    found = collections.Counter()
    for offset in offsets:
        for record in _chembl({**query, "limit": page, "offset": int(offset)})["activities"]:
            found[record.get("assay_variant_mutation")] += 1
    rows = [{"mutation": m, "measurements seen": n} for m, n in found.most_common()
            if m and m != "UNDEFINED MUTATION"]
    if not rows:
        print(f"no annotated mutants for {target} — the wild-type protein is all there is.")
    return pd.DataFrame(rows)


def fetch_target(name, max_records=3000, cache=True):
    """Every measured affinity for one target, collapsed to one row per molecule.

    Binding assays only, IC50/Ki/Kd only, and only records for which ChEMBL has computed
    a pChEMBL value. Repeat measurements of the same molecule become their median.
    """
    if name not in TARGETS:
        raise KeyError(f"{name!r} is not in TARGETS — add it, or pick one of: "
                       f"{', '.join(TARGETS)}")
    target, mutation = TARGETS[name]
    path = "chembl_" + name.replace(" ", "_").replace(",", "") + ".csv"

    if cache and os.path.exists(path):
        frame = pd.read_csv(path)
        print(f"{name}: {len(frame)} molecules (re-read from {path})")
    else:
        query = {"target_chembl_id": target, "pchembl_value__isnull": "false",
                 "standard_type__in": "IC50,Ki,Kd", "assay_type": "B",
                 "only": CHEMBL_FIELDS}
        query["assay_variant_mutation__isnull" if mutation is None
              else "assay_variant_mutation"] = "true" if mutation is None else mutation

        total = _chembl({**query, "limit": 1})["page_meta"]["total_count"]
        if total == 0:
            print(f"{name}: ChEMBL has no usable measurements. "
                  f"{NO_USABLE_MUTANT.get(name, '')}")
            return pd.DataFrame()

        page = 1000
        pages = int(np.ceil(min(total, max_records) / page))
        offsets = (np.linspace(0, max(total - page, 0), pages).astype(int)
                   if pages > 1 else [0])
        print(f"{name}: ChEMBL has {total} measurements, reading {len(offsets)} page(s)")
        records = []
        for offset in offsets:
            records += _chembl({**query, "limit": page, "offset": int(offset)})["activities"]

        raw = pd.DataFrame(records).dropna(subset=["canonical_smiles", "pchembl_value"])
        frame = (raw.astype({"pchembl_value": float})
                 .groupby("molecule_chembl_id")
                 .agg(smiles=("canonical_smiles", "first"),
                      pchembl=("pchembl_value", "median"),
                      n_measurements=("pchembl_value", "size"))
                 .reset_index())
        frame.to_csv(path, index=False)
        print(f"{name}: {len(raw)} measurements -> {len(frame)} molecules, cached in {path}")

    frame["mol"] = frame["smiles"].apply(Chem.MolFromSmiles)
    return frame.dropna(subset=["mol"]).reset_index(drop=True)


def descriptor_matrix(molecules):
    """The same five descriptors as Part 1, for any list of molecules."""
    return pd.DataFrame([[Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                          Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)]
                         for m in molecules], columns=DESCRIPTORS)


def fingerprint_matrix(molecules):
    """2048-bit Morgan fingerprints as a plain numeric array (one byte per bit)."""
    return np.array([np.array(morgan.GetFingerprint(m)) for m in molecules], dtype=np.uint8)


def nearest_training_similarity(test_molecules, training_molecules):
    """For each test molecule: the Tanimoto similarity to its closest training neighbour."""
    training = [morgan.GetFingerprint(m) for m in training_molecules]
    return np.array([max(DataStructs.BulkTanimotoSimilarity(morgan.GetFingerprint(m), training))
                     for m in test_molecules])


def diagonal(ax, values, margin=1.0, minimum_span=3.0):
    """The no-selectivity line and a band for 'within 10x', scaled to the data."""
    low, high = float(np.min(values)) - margin, float(np.max(values)) + margin
    if high - low < minimum_span:                  # keep a readable window
        middle = (low + high) / 2
        low, high = middle - minimum_span / 2, middle + minimum_span / 2
    ax.plot([low, high], [low, high], color="0.3", lw=1.2, zorder=1,
            label="equally active on both")
    ax.fill_between([low, high], [low - 1, high - 1], [low + 1, high + 1],
                    color="0.85", alpha=0.5, zorder=0, label="within 10x")
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_aspect("equal")


def label_points(ax, xs, ys, names, bold=()):
    """Name every point, nudging the labels apart so that none sits on another."""
    low, high = ax.get_ylim()
    gap, sideways = (high - low) / 18, (ax.get_xlim()[1] - ax.get_xlim()[0]) / 28
    settled, previous = {}, -np.inf
    for i in sorted(range(len(names)), key=lambda k: ys[k]):
        previous = settled[i] = max(ys[i], previous + gap)
    for i, name in enumerate(names):
        ax.annotate(name, xy=(xs[i], ys[i]), xytext=(xs[i] + sideways, settled[i]),
                    fontsize=10, va="center", zorder=6,
                    fontweight="bold" if name in bold else "normal",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                              edgecolor="none", alpha=0.75),
                    arrowprops=dict(arrowstyle="-", color="0.55", lw=0.8,
                                    shrinkA=0, shrinkB=5))


print("plumbing ready.")

# Part 1 — Can this molecule reach the brain?

For a drug aimed at the brain, crossing the **blood–brain barrier** is non-negotiable; for most
other drugs crossing it is a side-effect problem. Predicting it is a classic ADMET task, and the
dataset is small enough to break on purpose — which is what we will spend most of Part 1 doing.

In [ ]:
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(url)

df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
print(f"SMILES that RDKit could not parse: {df['mol'].isna().sum()}")
df = df.dropna(subset=["mol"]).reset_index(drop=True)

print(f"\n{len(df)} molecules")
print(df["p_np"].value_counts().rename({1: "crosses", 0: "does not"}).to_string())
print(f"\nmajority class: {max(df['p_np'].mean(), 1 - df['p_np'].mean()):.0%} "
      f"<- the accuracy you get for free, by always answering 'crosses'")

**76 % of this dataset crosses.** Keep that number: it is the score a model gets by learning
nothing at all. And before modelling, look at the molecules — a model you cannot trace back to
structures is a model you cannot defend.

In [ ]:
sample = pd.concat([df[df["p_np"] == 1].sample(4, random_state=1),
                    df[df["p_np"] == 0].sample(4, random_state=1)])

Draw.MolsToGridImage(
    list(sample["mol"]), molsPerRow=4, subImgSize=(260, 220),
    legends=[f"{name[:22]}\n{'CROSSES' if p else 'does NOT cross'}"
             for name, p in zip(sample["name"], sample["p_np"])])

## Structures into numbers — the "S" in QSAR

Five descriptors: one for size, one for greasiness, three for polarity.

In [ ]:
DESCRIPTORS = ["MW", "LogP", "TPSA", "NumHDonors", "NumHAcceptors"]

df[DESCRIPTORS] = df["mol"].apply(lambda m: pd.Series([
    Descriptors.MolWt(m),           # size
    Descriptors.MolLogP(m),         # lipophilicity
    Descriptors.TPSA(m),            # polar surface area
    Descriptors.NumHDonors(m),      # H-bond donors
    Descriptors.NumHAcceptors(m),   # H-bond acceptors
]))

df[DESCRIPTORS].describe().round(1)

Four of these five are **counts or sums of pharmacophore features** — donors, acceptors, polar
surface, greasiness. The previous notebook and this one are asking the same question with
different tools.

## Before training anything: what does guessing give you?

⏳ *The next cell is quick, but the three after the exercises are not — see the notes there.*

Two things to beat. The first is the majority class — always answer "crosses", be right 76 % of
the time, understand nothing. The second is the oldest rule in CNS medicinal chemistry: a
molecule with too much polar surface does not diffuse through a lipid membrane. One inequality,
no training, no data split.

Three scores come back, and they disagree with each other on purpose.

* **accuracy** — the fraction of molecules called correctly. One number, easily flattered by a
  big class.
* **balanced accuracy** — how much of each class you catch, averaged. Equivalently: the accuracy
  you would have got if both classes were the same size. **0.5 is the floor**, not 0, because
  always answering the same thing catches all of one class and none of the other.
* **MCC** — the ordinary correlation coefficient, applied to yes/no answers. +1 tracks the truth
  perfectly, 0 tracks nothing, −1 is perfectly backwards. It reads all four boxes of the
  confusion matrix, so a model that gets the big class right and ignores the small one cannot
  score well.

In [ ]:
TPSA_CUT = 90                                     # Å², the usual CNS rule of thumb

rule = (df["TPSA"] < TPSA_CUT).astype(int)
always_yes = np.ones(len(df), dtype=int)

for label, guess in [("always 'crosses'", always_yes), (f"TPSA < {TPSA_CUT} Å²", rule)]:
    tn, fp, fn, tp = confusion_matrix(df["p_np"], guess).ravel()
    print(f"--- {label} ---")
    print(f"   of the {tn + fp} that do NOT cross, it catches {tn:4d}  ({tn / (tn + fp):.0%})")
    print(f"   of the {fn + tp} that DO cross,     it catches {tp:4d}  ({tp / (fn + tp):.0%})")
    print(f"   accuracy           {accuracy_score(df['p_np'], guess):.3f}")
    print(f"   balanced accuracy  {balanced_accuracy_score(df['p_np'], guess):.3f}")
    print(f"   MCC                {matthews_corrcoef(df['p_np'], guess):.3f}\n")

figure, ax = plt.subplots(figsize=(9, 4.2))
bins = np.linspace(0, 250, 50)
for value, colour, label in [(1, "#d1495b", "crosses"), (0, "#4477aa", "does not cross")]:
    ax.hist(df.loc[df["p_np"] == value, "TPSA"], bins=bins, alpha=0.6,
            color=colour, label=label)
ax.axvline(TPSA_CUT, color="black", lw=1.6, ls="--")
ax.annotate(f"  TPSA = {TPSA_CUT} Å²", (TPSA_CUT, ax.get_ylim()[1] * 0.9))
ax.set_xlabel("topological polar surface area (Å²)")
ax.set_ylabel("molecules")
ax.set_title("one descriptor, one threshold")
ax.legend()
plt.show()

Read the two blocks against each other, and notice how differently the three scores separate
them.

| | accuracy | balanced accuracy | MCC |
|---|---|---|---|
| always "crosses" | 0.765 | 0.500 | 0.000 |
| TPSA < 90 Å² | 0.797 | **0.736** | **0.457** |
| the rule's advantage | +0.03 | +0.24 | +0.46 |

**On accuracy the rule looks almost worthless** — three points, 66 molecules out of 2 039. On
that column alone you would say the chemistry added nothing.

**The other two columns say it added everything.** One of these models detects 62 % of the
molecules that do not cross. The other detects none at all — a whole row of the confusion matrix
empty — and still scores 0.765, because 76 % of this dataset crosses and it gets every one of
those right for free. That is what balanced accuracy and MCC are for.

The one-line rule gets **0.74 balanced accuracy**. Remember that figure. In a few cells a random
forest on five descriptors will reach 0.77, and you will be able to say exactly how much the
machine learning was worth.

One aside for later: **that 76 % is not a fact about the blood–brain barrier.** Exercise 3.

## Training a model

Split the data, fit a random forest, predict the held-out part.

In [ ]:
X = df[DESCRIPTORS]
y = df["p_np"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"trained on {len(X_train)} molecules, testing on {len(X_test)}")

## Evaluation — and why accuracy is a trap

The same three scores as before, plus one more: **ROC-AUC**, the chance that a randomly chosen
penetrator is ranked above a randomly chosen non-penetrator. The helper prints each one next to
the score for always answering with the majority class.

In [ ]:
def evaluate(model, X_test, y_test, title=""):
    """Classification metrics, each next to the score of knowing nothing."""
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    majority = max(y_test.mean(), 1 - y_test.mean())

    print(f"--- {title} ---")
    print(f"accuracy                {accuracy_score(y_test, predicted):.3f}")
    print(f"   ... always-'yes'     {majority:.3f}   <- the score to beat")
    print(f"balanced accuracy       {balanced_accuracy_score(y_test, predicted):.3f}")
    print(f"ROC-AUC                 {roc_auc_score(y_test, probability):.3f}")
    print(f"MCC                     {matthews_corrcoef(y_test, predicted):.3f}")

    matrix = ConfusionMatrixDisplay(confusion_matrix(y_test, predicted),
                                    display_labels=["does not cross", "crosses"])
    matrix.plot(colorbar=False)
    plt.title(title)
    plt.show()
    return predicted


predicted = evaluate(model, X_test, y_test, "random split")

0.87 accuracy against a 0.77 baseline. Now compare it with the rule of thumb — carefully, because
the two numbers above were measured on different things. The forest's 0.774 is on the 408
held-out molecules; the rule's 0.736 was on all 2 039. Scored on the same 408, the rule gets
**0.730**. So the honest comparison is 0.774 against 0.730: the forest is ahead by four points.

Then look at where those four points come from, because it is not where you would want.

| on the same 408 test molecules | catches "crosses" | catches "does not" | balanced |
|---|---|---|---|
| TPSA < 90 Å² | 0.846 | **0.615** | 0.730 |
| random forest | 0.955 | **0.594** | 0.774 |

**The forest is slightly worse than one inequality at finding the molecules that do not cross.**
Every point it gained, it gained on the easy class. And "does not cross" is exactly the class you
want when you are filtering a library — you are looking for the compounds to throw out.

**Which error is worse here?** A false negative discards a good molecule — a missed shot. A false
positive sends a molecule that cannot reach its target into months of synthesis and testing. A
wasted shot costs more than a missed one, so for a brain drug the false positive is worse.

## What did the model learn?

In [ ]:
importance = pd.DataFrame({"descriptor": DESCRIPTORS,
                           "importance": model.feature_importances_}).sort_values("importance")

plt.barh(importance["descriptor"], importance["importance"], color="#4477aa")
plt.xlabel("importance")
plt.title("Which descriptors predict BBB penetration?")
plt.show()

**TPSA on top, then LogP** — polarity and greasiness, exactly the two quantities the rule of thumb
used. The model rediscovered passive membrane permeation from data alone. That is a genuine QSAR
result and the part of this exercise you could defend in a presentation.

The next two exercises are the part you could not.

### Exercise 1 — Split the data like a medicinal chemist

A random split puts close analogues of the same series on both sides of the line, so the model can
"recognise" a test molecule whose sibling it memorised. Real use is never like that: you apply a
model to a **new series**. The standard imitation is a **scaffold split** — group molecules by
their Bemis–Murcko framework and keep every scaffold entirely on one side.

Build the split, train the same model, evaluate with the same function, then compare the two sets
of numbers.

```python
def scaffold_split(frame, test_fraction=0.2):
    """Split so that no Bemis-Murcko scaffold appears in both parts."""
    scaffolds = {}
    for i, mol in enumerate(frame["mol"]):
        core = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        scaffolds.setdefault(core, []).append(i)

    groups = sorted(scaffolds.values(), key=len, reverse=True)
    n_test = int(len(frame) * test_fraction)
    test_rows, train_rows = [], []
    for group in groups:
        (test_rows if len(test_rows) + len(group) <= n_test else train_rows).extend(group)
    return train_rows, test_rows
```

<details>
<summary>🔍 Solution — and a result worth remembering</summary>

```python
train_rows, test_rows = scaffold_split(df)
scaffold_model = RandomForestClassifier(n_estimators=100, random_state=42)
scaffold_model.fit(X.iloc[train_rows], y.iloc[train_rows])

evaluate(scaffold_model, X.iloc[test_rows], y.iloc[test_rows], "scaffold split")
```

| | random split | scaffold split |
|---|---|---|
| accuracy | 0.87 | 0.83 |
| always-"yes" baseline | 0.77 | **0.86** |
| balanced accuracy | 0.77 | **0.57** |
| ROC-AUC | 0.90 | 0.67 |
| MCC | 0.61 | **0.18** |

**Accuracy hardly moved.** On that number alone you would conclude the model transfers fine to new
chemistry.

**Everything else collapsed.** Balanced accuracy 0.57, a coin flip; MCC 0.18.

Note where the two baselines come from, because they are not the same number measured twice.
Each is the majority class **of its own test set**: the random test set is 77 % penetrators, like
the dataset as a whole, but the scaffold test set is **86 %**. The split fills the test half with
the largest scaffold groups first, and the largest groups here are steroids, phenothiazines,
benzodiazepines and barbiturates — CNS drug classes, where almost everything crosses. Eight
scaffolds account for all 407 test molecules; the other 1 017 scaffolds stay in training.

So 0.83 accuracy is now *below* the 0.86 you would get by answering "yes" every time. **On these
held-out classes the model is worse than useless** — and being specific about *which* held-out
classes is part of the result, not a detail.

1. **A random split overestimates performance on any dataset built from analogue series** — which
   is nearly every medicinal chemistry dataset.
2. **Always quote a baseline.** A score without "what would guessing give?" is not a result.
3. **Validate the way the model will be used.** New scaffolds in production, new scaffolds in the
   test set.

</details>

### Exercise 2 — Four molecules, four different problems

Numbers tell you how much is wrong; structures tell you what is wrong. Predict these four by hand
before you run the cell, then see how confident the model is about each.

| | | your guess |
|---|---|---|
| **caffeine** | everyone's positive control | |
| **glucose** | far too polar to diffuse — but the brain runs on it | |
| **L-DOPA** | the Parkinson's drug; dopamine itself does not get in, L-DOPA does | |
| **loperamide** | greasy, permeable-looking anti-diarrhoeal that is deliberately *not* a CNS drug | |

In [ ]:
PROBES = {
    "caffeine":   "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "glucose":    "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
    "L-DOPA":     "N[C@@H](Cc1ccc(O)c(O)c1)C(=O)O",
    "loperamide": "CN(C)C(=O)C(CCN1CCC(O)(c2ccc(Cl)cc2)CC1)(c1ccccc1)c1ccccc1",
}

probe_mols = {name: Chem.MolFromSmiles(s) for name, s in PROBES.items()}
probes = descriptor_matrix(probe_mols.values())
probes.insert(0, "molecule", list(probe_mols))
probes["P(crosses)"] = model.predict_proba(probes[DESCRIPTORS])[:, 1].round(2)

# what does the dataset itself say about these four?
WORDS = {(0,): "does not cross", (1,): "crosses",
         (0, 1): "BOTH — the dataset contradicts itself"}
canonical = {Chem.MolToSmiles(m): name for name, m in probe_mols.items()}
df["canonical"] = df["mol"].apply(Chem.MolToSmiles)
labelled = df[df["canonical"].isin(canonical)]
probes["BBBP says"] = probes["molecule"].map(
    labelled.groupby(labelled["canonical"].map(canonical))["p_np"]
    .apply(lambda s: WORDS[tuple(sorted(set(s)))])).fillna("not in the dataset")

display(probes.round(2))
display(Draw.MolsToGridImage(list(probe_mols.values()), molsPerRow=4, subImgSize=(250, 210),
                             legends=[f"{n}  P={p:.2f}" for n, p in
                                      zip(probes["molecule"], probes["P(crosses)"])]))

Caffeine: 0.97, confident and right. The other three sit between 0.4 and 0.6 — the model is not
confidently wrong, it is **undecided**, which is its way of saying the answer is not in these five
numbers.

* **Glucose** crosses, on a dedicated transporter (GLUT1). Its descriptors say "far too polar",
  and by passive diffusion they are right.
* **L-DOPA** crosses on the LAT1 amino-acid transporter — the entire reason it is the drug and
  dopamine is not. Same story: the mechanism is not a descriptor.
* **Loperamide** looks ideal and does not get in, because P-glycoprotein pumps it straight back
  out. Invisible for the opposite reason.

Now look at the last column.

### Exercise 3 — Where do the labels sit?

Two of those molecules carry **both labels**. Before hunting for more, do something simpler: the
file has a row number, so look at how the labels are distributed down it.

<details>
<summary>🔍 Solution</summary>

```python
order = df.sort_values("num")
print("last row anywhere in the file carrying label 0:",
      int(order.loc[order["p_np"] == 0, "num"].max()))
display(order.groupby((order["num"] // 250) * 250)["p_np"]
        .agg(molecules="size", fraction_crossing="mean").round(2))
```

**BBBP is two lists glued together.** The last `0` anywhere in the file is row 948. Everything
after it — 1 103 molecules, more than half the dataset — is a penetrator. The first block is
49 % / 51 %; the second can only say yes. The two halves even capitalise their compound names
differently, which is the fingerprint of two people compiling two lists.

That explains the contradictions. Nine of the ten pair an early row with a late one:

```python
clash = df.groupby("canonical").filter(lambda g: g["p_np"].nunique() > 1)
for _, group in clash.groupby("canonical"):
    print("   ", ", ".join(f"#{int(n)} {nm} -> {p}"
                           for n, nm, p in zip(group["num"], group["name"], group["p_np"])))
```

`#102 acetylsalicylate → 0` against `#1009 aspirin → 1`; `#284 levodopa → 0` against
`#1881 levodopa → 1`. The late label is `1` **only because that block contains nothing else**, so
it carries no information at all. These are not two measurements disagreeing — they are compounds
from a curated list reappearing in a positives-only one. (The one real disagreement is
`#17 BRL53080 → 1` against `#552 loperamide → 0`, both inside the first block.)

Two consequences, and the second is the bigger one:

1. **There is a ceiling.** With self-contradictory rows and an unknown number simply wrong, the
   difference between a 0.86 model and a 0.88 model on this benchmark is probably noise.
2. **The 76 % is an artefact of assembly, not a fact about the brain.** The imbalance that makes
   accuracy useless here — the thing this whole section is built around — exists because somebody
   concatenated a positives-only list onto a balanced one. Hundreds of papers quote accuracies on
   the result.

</details>

## Descriptors or fingerprints?

Five numbers chosen by a chemist, or 2048 bits that just record which substructures are present?
Same model, same splits, both feature sets.

> ⏳ **Start the next cell now** — four forests on 2 048 features, around half a minute.

In [ ]:
fingerprints = fingerprint_matrix(df["mol"])
print(f"fingerprint matrix: {fingerprints.shape[0]} molecules x {fingerprints.shape[1]} bits, "
      f"{fingerprints.sum(axis=1).mean():.0f} bits set per molecule on average")


def scaffold_split(frame, test_fraction=0.2):
    """Split so that no Bemis-Murcko scaffold appears in both parts."""
    scaffolds = {}
    for i, mol in enumerate(frame["mol"]):
        core = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        scaffolds.setdefault(core, []).append(i)
    groups = sorted(scaffolds.values(), key=len, reverse=True)
    n_test = int(len(frame) * test_fraction)
    test_rows, train_rows = [], []
    for group in groups:
        (test_rows if len(test_rows) + len(group) <= n_test else train_rows).extend(group)
    return train_rows, test_rows


random_rows = train_test_split(np.arange(len(df)), test_size=0.2,
                               random_state=42, stratify=y)
splits = {"random": random_rows, "scaffold": scaffold_split(df)}
features = {"5 descriptors": df[DESCRIPTORS].values, "2048-bit fingerprint": fingerprints}

rows = []
jobs = [(split_name, feature_name) for split_name in splits for feature_name in features]
for split_name, feature_name in tqdm(jobs, desc="training"):
    train, test = splits[split_name]
    matrix = features[feature_name]
    forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    forest.fit(matrix[train], y.iloc[train])
    predicted = forest.predict(matrix[test])
    rows.append({"split": split_name, "features": feature_name,
                 "balanced accuracy": balanced_accuracy_score(y.iloc[test], predicted),
                 "ROC-AUC": roc_auc_score(y.iloc[test],
                                          forest.predict_proba(matrix[test])[:, 1]),
                 "MCC": matthews_corrcoef(y.iloc[test], predicted)})

comparison = pd.DataFrame(rows)
display(comparison.round(3).set_index(["split", "features"]))

figure, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
for ax, metric in zip(axes, ["balanced accuracy", "ROC-AUC", "MCC"]):
    table = comparison.pivot(index="split", columns="features", values=metric)
    table.plot.bar(ax=ax, color=["#4477aa", "#d1495b"], rot=0, legend=(ax is axes[0]),
                   width=0.7)
    ax.set_title(metric)
    ax.set_xlabel("")
axes[0].axhline(0.5, color="black", lw=1, ls=":")
plt.tight_layout()
plt.show()

Not the clean story you might expect, so read it as it is.

**On the random split the fingerprint wins clearly** (MCC 0.69 against 0.57). **On the scaffold
split both collapse, and the fingerprint collapses harder** — its MCC falls below the
descriptors'.

That is the same lesson as Exercise 1, seen from the other side. A fingerprint is a list of
substructures, so it is very good at recognising a molecule it has seen the relatives of, and has
nothing to say about a scaffold it has never met. Five physical descriptors transfer badly too,
but they at least mean the same thing on new chemistry.

**So which should you use?** It depends on what the endpoint depends on. BBB penetration is a
*physical property* — polarity and size — and those are what the descriptors measure directly.
Binding to a particular protein is a *shape-and-substructure* question. Part 2 is a binding
endpoint, and there the answer comes out very differently.

# Part 2 — Your own target

Everything above was practice on a dataset that comes pre-packaged. Real measurements do not. From
here on you work on **your project's protein**, with the data the field has actually published,
and the figure that comes out is yours to present.

The endpoint changes too. BBB penetration was yes/no; binding affinity is a **number**, so Part 2
is a regression. We model **pChEMBL = −log₁₀(IC₅₀ or Kᵢ in molar)**. The minus sign is the part
to hold on to: **a bigger pChEMBL means a stronger binder.** 6 is micromolar and unremarkable, 9
is nanomolar and a serious drug candidate, and one unit is one order of magnitude — so a
difference of 2 between two proteins is a hundred-fold preference.

| your track | target A | target B | the question the figure answers |
|---|---|---|---|
| **selectivity** | one protein | its relative | does my drug prefer one of them, and would a model have told me? |
| **resistance** | wild type | clinical mutant | does the drug lose the mutant, and does the next-generation one win it back? |

A few cells from now you will have trained two models and plotted your drug on both. Everything
after that is about whether to believe the plot.

In [ ]:
available = pd.DataFrame([{"name": name, "ChEMBL target": target,
                           "mutation": mutation or "— (wild type)"}
                          for name, (target, mutation) in TARGETS.items()])
display(available.set_index("name"))

Two things about that table before you use it.

**"EGFR T790M" fetches the assays annotated `L858R,T790M`.** In ChEMBL a mutation is a property of
the **assay**, not a separate protein entry — the same target id with a flag saying which variant
was in the tube. And almost nobody measures T790M alone, because in patients it arises on top of
the activating L858R mutation. The database records what was measured, which is the clinical
double mutant: 1 900 measurements against 280 for T790M by itself.

**Your target may not be in the table.** Two helpers find what you need:

```python
find_target("JAK2")            # -> CHEMBL2971, Tyrosine-protein kinase JAK2
find_mutations("CHEMBL2971")   # -> which mutants have actually been measured, and how often
```

Then add one line, and it works like any other entry:

```python
TARGETS["JAK2"]       = ("CHEMBL2971", None)
TARGETS["JAK2 V617F"] = ("CHEMBL2971", "V617F")
```

`find_mutations` matters more than it looks. A mutation lives on the assay record, so there is no
page on the ChEMBL website that lists the available variants for a protein — without it you would
have to guess the exact string, and `"T790M"` would silently have given you a seventh of the data
that `"L858R,T790M"` does.

### Your four lines

Everything downstream reads these. Nothing else in Part 2 needs editing.

> ⏳ **Edit the next cell, then start the one after it straight away** — it downloads both
> targets from ChEMBL, about a minute, and caches them so a re-run is instant. If you see
> `ChEMBL attempt 1/3 failed`, ignore it: the server is busy and the cell retries by itself.

In [ ]:
MY_DRUG   = "gefitinib"                                  # <-- your compound
TARGET_A  = "EGFR"                                       # <-- protein A, or the wild type
TARGET_B  = "EGFR T790M"                                 # <-- protein B, or the mutant
SHORTLIST = ["erlotinib", "lapatinib", "osimertinib"]    # <-- from Exercise 7, or [] for none

In [ ]:
data = {name: fetch_target(name) for name in (TARGET_A, TARGET_B)}

summary = []
for name, frame in data.items():
    if frame.empty:
        print("\n" + "=" * 72)
        print(f"STOP — no usable data for {name}. Change TARGET_A / TARGET_B above and")
        print("re-run this cell; nothing below will work until both targets have data.")
        print("=" * 72)
        continue
    cores = {MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False)
             for m in frame["mol"]}
    summary.append({"target": name, "molecules": len(frame), "scaffolds": len(cores),
                    "per scaffold": len(frame) / len(cores),
                    "pChEMBL min": frame["pchembl"].min(),
                    "median": frame["pchembl"].median(),
                    "max": frame["pchembl"].max(),
                    "spread (sd)": frame["pchembl"].std()})

display(pd.DataFrame(summary).round(2).set_index("target"))

figure, ax = plt.subplots(figsize=(9, 4))
for (name, frame), colour in zip(data.items(), ["#4477aa", "#d1495b"]):
    if not frame.empty:
        ax.hist(frame["pchembl"], bins=np.arange(3.5, 12, 0.25), alpha=0.55,
                color=colour, label=f"{name}  (n = {len(frame)})")
ax.set_xlabel("pChEMBL   (6 = µM, 9 = nM)")
ax.set_ylabel("molecules")
ax.set_title("what has actually been measured against your two proteins")
ax.legend()
plt.show()

for name, frame in data.items():
    if not frame.empty and len(frame) < 200:
        print(f"!! {name} has only {len(frame)} molecules. Train the model anyway, but treat "
              f"every number it produces as a hint. Fewer than ~100 and QSAR cannot answer "
              f"your question at all — which is a legitimate finding, and the reason the next "
              f"notebook is about docking.")

Four things to read off that before modelling anything.

1. **How much data is there?** A few hundred molecules supports a weak model; a few thousand
   supports a useful one. Under about a hundred, stop — write "insufficient public data" on the
   slide and move to structure-based methods. That is a result, not a failure.
2. **How many molecules per scaffold?** Around 3 means the data is a handful of optimised series
   that a model can interpolate inside. Close to 1 means every molecule is its own scaffold and
   there is nothing to interpolate *from*. Note the number; two cells below you will see how
   little it actually guarantees.
3. **How wide is the spread?** The standard deviation is the error a model that always predicts
   the mean would make. Around 1.2 log units is typical, and it is the number to beat.
4. **Are the two histograms in the same place?** If the mutant set is shifted, part of what you
   are about to see is which compounds people chose to test, not only how the protein changed.

## Does the pipeline transfer to binding data?

Same four combinations as on BBBP — descriptors or fingerprints, random or scaffold split — now
on a binding endpoint. The endpoint is a number, so the scores change:

* **R²** — how much better than quoting the average this is. 1 is perfect, 0 is no better than
  the average, negative is worse than that. 0.67 means it accounts for two thirds of the
  variation the average leaves unexplained. It does **not** tell you whether the model is good
  enough for your purpose.
* **MAE** — mean absolute error: how many log units a typical prediction is off by. This is the
  one to read for "good enough". 0.5 means a typical answer is wrong by about a factor of three.
* **MAE of mean-model** — the same, for a model that ignores the structure entirely and always
  answers the training average. It is the number MAE has to beat.

Every R² here is measured on **held-out** molecules. Fit a random forest to all the data and ask
for its R² on that same data and you will get something close to 1, which means nothing at all.

> ⏳ **Start the next cell now** — four forests, around half a minute.

In [ ]:
def evaluate_regression(model, X_test, y_test, label=""):
    """Regression metrics, each next to the score of predicting the training mean."""
    predicted = model.predict(X_test)
    return {"model": label,
            "R2": r2_score(y_test, predicted),
            "MAE": mean_absolute_error(y_test, predicted),
            "MAE of mean-model": float(np.mean(np.abs(y_test - np.mean(y_test))))}


frame_a = data[TARGET_A]
splits = {"random": train_test_split(np.arange(len(frame_a)), test_size=0.2, random_state=42),
          "scaffold": scaffold_split(frame_a)}
features = {"5 descriptors": descriptor_matrix(frame_a["mol"]).values,
            "2048-bit fingerprint": fingerprint_matrix(frame_a["mol"])}

rows = []
jobs = [(split_name, feature_name) for split_name in splits for feature_name in features]
for split_name, feature_name in tqdm(jobs, desc="training"):
    train, test = splits[split_name]
    matrix = features[feature_name]
    # max_features: scikit-learn's regressor searches every one of the 2048 bits at each
    # split by default, which is ten times slower here and no more accurate.
    forest = RandomForestRegressor(n_estimators=300, max_features=0.25,
                                   random_state=42, n_jobs=-1)
    forest.fit(matrix[train], frame_a["pchembl"].iloc[train])
    row = evaluate_regression(forest, matrix[test], frame_a["pchembl"].iloc[test].values)
    rows.append({"split": split_name, "features": feature_name, **row})

binding = pd.DataFrame(rows).drop(columns="model")
display(binding.round(3).set_index(["split", "features"]))

figure, ax = plt.subplots(figsize=(7, 4))
binding.pivot(index="split", columns="features", values="R2").plot.bar(
    ax=ax, color=["#4477aa", "#d1495b"], rot=0, width=0.7)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("R²")
ax.set_title(f"predicting affinity against {TARGET_A}")
plt.tight_layout()
plt.show()

On a binding endpoint **the fingerprint wins every time**, and usually by a wide margin. On
wild-type EGFR the scaffold-split R² is 0.67 for the fingerprint against 0.08 for the descriptors
— the latter meaning you would do about as well by ignoring the structure and quoting the average.

That is not a flaw in the descriptors; it is what they are. Molecular weight, LogP and TPSA
describe a molecule's *bulk physical behaviour*, which is the right language for crossing a
membrane and the wrong one for fitting a particular pocket. Two molecules with identical MW, LogP
and TPSA can differ a thousand-fold against the same kinase. The fingerprint at least records
**which groups are where**.

**How good the fingerprint gets depends entirely on your target**, so read your own table rather
than ours. Three real examples — same pipeline, same sample size, all scaffold-split:

| target | molecules | scaffolds | per scaffold | descriptors R² | fingerprint R² |
|---|---|---|---|---|---|
| EGFR | 2 182 | 833 | 2.6 | 0.08 | **0.67** |
| ABL1 | 1 985 | 805 | 2.5 | 0.51 | **0.73** |
| AChE | 2 369 | 1 057 | 2.2 | −0.09 | **0.17** |

AChE has the **most** data of the three and much the worst model. Its molecules are spread over
the most scaffolds, which is part of the story — hold those out and less is left to interpolate
from — but 2.2 against 2.6 is a small difference for a gap between 0.17 and 0.67, so it is not
the whole of it. Nothing in the first four columns would have let you predict the last one.

**That is the practical point.** You cannot tell in advance whether QSAR will work on your target.
You run the scaffold split and look. R² 0.17 is not a model, and finding that out in five minutes
is worth considerably more than a figure you half believe.

## The figure: one drug, two proteins

Train one model per target, then ask both of them about your drug and your shortlist. A compound
on the diagonal is equally active on both; distance from the diagonal **is** selectivity.

> ⏳ **Start the next cell now** — fingerprinting both libraries, two forests and a few PubChem
> lookups, around half a minute.

In [ ]:
# fingerprint each library once; everything below reuses these
bits, prints = {}, {}
for name, frame in tqdm(list(data.items()), desc="fingerprinting"):
    prints[name] = [morgan.GetFingerprint(m) for m in frame["mol"]]
    bits[name] = np.array([np.array(f) for f in prints[name]], dtype=np.uint8)


def fit_target(name, without=None, cutoff=0.7):
    """A fingerprint random forest for one target.

    `without` is a molecule to erase from the training set along with every analogue at or
    above `cutoff` Tanimoto similarity to it. Returns the model and how many rows went.
    """
    frame = data[name]
    keep = np.ones(len(frame), dtype=bool)
    if without is not None:
        keep = np.array(DataStructs.BulkTanimotoSimilarity(
            morgan.GetFingerprint(without), prints[name])) < cutoff
    forest = RandomForestRegressor(n_estimators=300, max_features=0.25,
                                   random_state=42, n_jobs=-1)
    forest.fit(bits[name][keep], frame["pchembl"].values[keep])
    return forest, int((~keep).sum())


probe_mols = {}
for name in tqdm([MY_DRUG] + list(SHORTLIST), desc="looking up compounds"):
    smiles = smiles_from_pubchem(name)
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    if mol is None:
        print(f"STOP — PubChem has no compound called {name!r}. Put its SMILES string in "
              f"place of the name and re-run.")
    else:
        probe_mols[name] = mol

models = {name: fit_target(name)[0] for name in data}
probe_matrix = fingerprint_matrix(probe_mols.values())

predictions = pd.DataFrame({"compound": list(probe_mols)})
for name in (TARGET_A, TARGET_B):
    predictions[f"predicted {name}"] = models[name].predict(probe_matrix)
    canon = {Chem.MolToSmiles(m): n for n, m in probe_mols.items()}
    measured = (data[name].assign(canonical=data[name]["mol"].apply(Chem.MolToSmiles))
                .query("canonical in @canon"))
    predictions[f"measured {name}"] = predictions["compound"].map(
        dict(zip(measured["canonical"].map(canon), measured["pchembl"])))

predictions["selectivity"] = (predictions[f"predicted {TARGET_B}"]
                              - predictions[f"predicted {TARGET_A}"])
display(predictions.round(2).set_index("compound"))

x = predictions[f"predicted {TARGET_A}"].values
z = predictions[f"predicted {TARGET_B}"].values

figure, ax = plt.subplots(figsize=(7.2, 7))
diagonal(ax, np.concatenate([x, z]))
low, high = ax.get_xlim()
ax.text(high - 0.2, low + 0.3, f"selective for\n{TARGET_A}", ha="right", va="bottom",
        fontsize=10, color="0.35")
ax.text(low + 0.2, high - 0.3, f"selective for\n{TARGET_B}", ha="left", va="top",
        fontsize=10, color="0.35")

is_drug = (predictions["compound"] == MY_DRUG).values
ax.scatter(x[~is_drug], z[~is_drug], s=140, color="#4477aa", zorder=3,
           edgecolor="white", linewidth=1.2, label="your shortlist")
ax.scatter(x[is_drug], z[is_drug], s=420, marker="*", color="#d1495b", zorder=4,
           edgecolor="white", linewidth=1.2, label=MY_DRUG)
label_points(ax, x, z, list(predictions["compound"]), bold={MY_DRUG})

ax.set_xlabel(f"predicted pChEMBL on {TARGET_A}")
ax.set_ylabel(f"predicted pChEMBL on {TARGET_B}")
ax.set_title("your compound and your shortlist, on both proteins")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

### How to read it

The grey band is "within one log unit", i.e. within a factor of ten — about the point where a
difference stops being an artefact of assay-to-assay variation and starts being chemistry. A
compound inside the band is, as far as these models can tell, **not selective**.

For the worked example the two models put gefitinib at 7.9 on wild-type EGFR and 6.2 on the
mutant: 1.7 log units down, a fifty-fold loss, which is what makes T790M a resistance mutation in
the clinic. Osimertinib goes the other way, 7.3 up to 8.4. **The two drugs sit on opposite sides
of the diagonal** — the whole point of designing a third-generation inhibitor, recovered from
published numbers.

Before you put that on a slide: the model has almost certainly seen both of those drugs during
training.

## Was that a prediction, or a memory?

Your drug is a famous compound measured against your target many times. It is in ChEMBL,
therefore in the training set, and a random forest can memorise a training row perfectly. Compare
the measured value with what the model says — if they agree to two decimal places, nothing was
predicted.

So remove it. And because its close analogues carry the same information, remove them too: every
training molecule at Tanimoto ≥ 0.7 to your drug.

> ⏳ **Start the next cell now** — it retrains the model once per compound per target, so it is
> the slowest cell in the notebook. A minute or two.

In [ ]:
CUTOFF = 0.7

rows = []
jobs = [(name, compound) for name in data for compound in probe_mols]
measured = predictions.set_index("compound")
for name, compound in tqdm(jobs, desc="retraining without each compound"):
    mol = probe_mols[compound]
    honest, removed = fit_target(name, without=mol, cutoff=CUTOFF)
    single = fingerprint_matrix([mol])
    rows.append({"target": name, "compound": compound,
                 "measured": measured.at[compound, f"measured {name}"],
                 "with analogues": models[name].predict(single)[0],
                 "analogues removed": honest.predict(single)[0],
                 "rows removed": removed})

leakage = pd.DataFrame(rows)
leakage["shift"] = leakage["analogues removed"] - leakage["with analogues"]
display(leakage.round(2).set_index(["target", "compound"]))

wide = leakage.pivot(index="compound", columns="target",
                     values=["with analogues", "analogues removed"])

figure, ax = plt.subplots(figsize=(7.2, 7))
diagonal(ax, wide.values)
for compound in wide.index:
    start = [wide.loc[compound, ("with analogues", t)] for t in (TARGET_A, TARGET_B)]
    end = [wide.loc[compound, ("analogues removed", t)] for t in (TARGET_A, TARGET_B)]
    colour = "#d1495b" if compound == MY_DRUG else "#4477aa"
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", color=colour, lw=1.8, alpha=0.85,
                                shrinkA=5, shrinkB=5))
    ax.scatter(*start, s=100, facecolor="white", edgecolor=colour, linewidth=1.8, zorder=3)
    ax.scatter(*end, s=380 if compound == MY_DRUG else 130,
               marker="*" if compound == MY_DRUG else "o", color=colour, zorder=4,
               edgecolor="white", linewidth=1.2)

label_points(ax, [wide.loc[c, ("analogues removed", TARGET_A)] for c in wide.index],
             [wide.loc[c, ("analogues removed", TARGET_B)] for c in wide.index],
             list(wide.index), bold={MY_DRUG})

ax.scatter([], [], s=100, facecolor="white", edgecolor="0.4", linewidth=1.8,
           label="analogues still in the training set")
ax.scatter([], [], s=130, color="0.4", label="analogues removed")
ax.set_xlabel(f"predicted pChEMBL on {TARGET_A}")
ax.set_ylabel(f"predicted pChEMBL on {TARGET_B}")
ax.set_title("the arrow is how much of the figure was memory")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

### What the arrows mean

Three numbers in that table decide whether your figure survives.

**`measured` against `with analogues`.** For the worked example: gefitinib measured 7.89 on
wild-type EGFR, predicted 7.90. The model did not predict that, it looked it up.

**The length of the arrow.** Every point moves when its analogues go, by 0.1 to 0.6 log units
here. That is the honest width of your error bar, and it is wider than any R² suggested.

**`rows removed`.** This is the number to look at first. Removing gefitinib's neighbours from the
wild-type set deletes 15 of 2 182 molecules and the prediction barely budges, by 0.11 — the
answer was supported by the whole series, not by one row. Removing osimertinib's neighbours from
the mutant set deletes 71, and the prediction falls 0.6 log units, because a large part of what
is known about that mutant *is* the osimertinib series.

In the worked example the conclusion survives: gefitinib still loses the mutant, osimertinib still
wins it, both by more than a log unit. The *direction* was real; the *precision* was not.

It does not always survive. On ABL1 T315I, removing imatinib and its **two** close analogues from
a 408-molecule set moves the prediction by 0.8 log units, and imatinib's resistance effect shrinks
from 1.3 log units to 0.4 — inside the grey band, which is to say gone. A prediction resting on
two training rows is not a prediction. If that is what your data does, say so: it is exactly what
the "three reasons our conclusion could be wrong" slide is for, and a far stronger slide than a
confident number.

## How far out can you trust it?

OECD principle 3 asks every QSAR model to state its **applicability domain** — the chemistry it is
entitled to an opinion about. The practical version: for each test molecule, how similar is its
nearest training neighbour? Then check whether the error grows as that similarity falls.

> ⏳ **Start the next cell now** — one more forest plus a similarity sweep, around half a minute.

In [ ]:
train_rows, test_rows = scaffold_split(frame_a)
domain_model = RandomForestRegressor(n_estimators=300, max_features=0.25,
                                     random_state=42, n_jobs=-1)
domain_model.fit(bits[TARGET_A][train_rows],
                 frame_a["pchembl"].iloc[train_rows])

nearest = nearest_training_similarity(frame_a["mol"].iloc[test_rows],
                                      frame_a["mol"].iloc[train_rows])
error = np.abs(domain_model.predict(bits[TARGET_A][test_rows])
               - frame_a["pchembl"].iloc[test_rows].values)

thirds = pd.qcut(nearest, 3, labels=["least similar third", "middle third",
                                     "most similar third"])
table = pd.DataFrame({"nearest neighbour": nearest, "absolute error": error, "third": thirds})
display(table.groupby("third", observed=False)
        .agg(molecules=("absolute error", "size"),
             mean_error=("absolute error", "mean"),
             similarity_from=("nearest neighbour", "min"),
             similarity_to=("nearest neighbour", "max")).round(2))

figure, ax = plt.subplots(figsize=(9, 4.6))
ax.scatter(nearest, error, s=18, alpha=0.45, color="#4477aa", label="one test molecule")
for third, group in table.groupby("third", observed=False):
    centre = group["nearest neighbour"].mean()
    ax.plot([group["nearest neighbour"].min(), group["nearest neighbour"].max()],
            [group["absolute error"].mean()] * 2, color="#d1495b", lw=3,
            label="mean error of this third" if third == "least similar third" else None)
    ax.annotate(f"{group['absolute error'].mean():.2f}", (centre, group["absolute error"].mean()),
                textcoords="offset points", xytext=(0, 8), ha="center", color="#d1495b",
                fontweight="bold")
ax.axhline(frame_a["pchembl"].std(), color="black", ls=":", lw=1.2,
           label="error of predicting the mean")
ax.set_xlabel("Tanimoto similarity to the nearest training molecule")
ax.set_ylabel("|predicted − measured|   (log units)")
ax.set_title(f"the applicability domain of your {TARGET_A} model")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The red bars should step **downwards** from left to right: the closer a molecule is to something
the model has seen, the better the prediction. In the worked example the least-similar third is
wrong by 0.87 log units and the other two by about 0.63 — and the dotted line shows that the
least-similar third is not far off simply quoting the average.

Two practical consequences.

* **A prediction comes with a similarity.** "pIC₅₀ 8.2" is not a result; "pIC₅₀ 8.2, nearest
  training neighbour 0.71" is. Quote both for every compound you carry into docking.
* **The trend is sometimes flat or noisy**, especially with few hundred molecules or when the
  whole test set is already far from the training data. That is informative too: it means the
  model is equally uncertain everywhere, and you should not trust it anywhere.

### Exercise 4 — Your figure, and what is wrong with it

Run everything above on your own four lines, then write the three sentences that go under the
figure in your presentation.

1. **The claim.** What does the diagonal plot say about your drug? Name the number — how many log
   units from the diagonal, and which side?
2. **The honest version.** How far did the arrow move it, and how many rows did you remove to find
   out? If your conclusion changed sign or shrank inside the grey band, that is your answer and it
   is a perfectly good one.
3. **The limit.** How many analogues did you have to remove, and how close was the closest? Your
   drug is in the training data, so it is not on the applicability-domain plot — this is the
   number that plays the same role for it.

   ```python
   for name in data:
       similarity = DataStructs.BulkTanimotoSimilarity(
           morgan.GetFingerprint(probe_mols[MY_DRUG]), prints[name])
       neighbours = sorted(similarity, reverse=True)[1:]      # drop the compound itself
       print(f"{name}: closest analogue {neighbours[0]:.2f}, "
             f"{sum(s >= 0.7 for s in neighbours)} above 0.7")
   ```

<details>
<summary>🔍 What a good answer looks like</summary>

> Gefitinib is predicted 1.7 log units below the diagonal on EGFR T790M — roughly
> fifty-fold weaker on the mutant — while osimertinib sits 1.2 log units above
> it. Removing each compound together with its analogues at Tanimoto ≥ 0.7 strips 15 molecules
> from the wild-type set for gefitinib and 71 from the mutant set for osimertinib, and moves the
> individual predictions by up to 0.6 log units — but the separations only go to
> 1.6 and 1.1, so neither sign changes. The direction of the effect does not
> depend on the two drugs being in the training data.
>
> How far to trust it is a separate question, and the two compounds are not equally placed.
> Osimertinib has 71 near neighbours in the mutant set, the closest at 0.91 Tanimoto. Gefitinib
> has **one**, at 0.70 — exactly the cutoff — which is why removing that single row moved its
> mutant prediction at all. A conclusion resting on one analogue is worth stating as such.

Four features of that paragraph worth copying: a number for every claim, each number attached to
the set it came from, the size of the test that could have falsified it, and one sentence naming
what the result does not cover.

Note what the paragraph does **not** claim. Neither drug appears on the applicability-domain plot
above — that plot is drawn from the scaffold-split *test* set, and both of these are in the
training half. Their nearest-neighbour similarities had to be measured separately.

</details>

### Exercise 5 — Three reasons this could be wrong

Your final talk needs one slide titled *three reasons our conclusion could be wrong*. Part 2 has
handed you more than three candidates; pick the ones that actually apply to **your** data and put
a number on each.

<details>
<summary>🔍 The strongest candidates, and the number that makes each concrete</summary>

| reason | the number to quote |
|---|---|
| the drug was in the training set | measured vs. predicted, and how many rows you removed |
| too little data on the mutant / second protein | n molecules, n scaffolds |
| the model only knows substructures, not the protein | descriptor R² against fingerprint R² |
| the prediction is outside the applicability domain | nearest-neighbour Tanimoto for your compound |
| what was measured is not what was asked | the two pChEMBL histograms — are the sets comparable? |
| published numbers mix assays | pChEMBL pools IC₅₀, Kᵢ and K_d from many labs |

The last one deserves a sentence of its own, because it is invisible in every plot above. An IC₅₀
depends on the substrate concentration used; a Kᵢ does not. Our fetch pooled both, across decades
and dozens of labs, and collapsed repeats to a median. That is standard practice and it is also
why 0.3 log units of apparent precision is not real.

</details>

# Part 3 — Where do your compounds sit?

The applicability-domain plot reduced "is this molecule like the training data?" to one number per
molecule. A map shows the same thing for a whole library at once — and shows you *why* a scaffold
split behaves the way it does.

UMAP squeezes 2048 fingerprint bits down to two so we can look at them. One setting matters more
than all the others:

**A fingerprint is a set of substructures, not a point in space.** The natural way to compare two
of them is the **Jaccard/Tanimoto** distance — the fraction of substructures they share — not the
Euclidean distance, which is dominated by the thousands of bits that are zero in both molecules.
`metric="jaccard"` is a one-word change that visibly improves the map.

> ⏳ **Start the next cell now** — UMAP on 2 000 molecules, around half a minute.

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="jaccard", random_state=42)
embedding = reducer.fit_transform(fingerprints)
df["UMAP1"], df["UMAP2"] = embedding[:, 0], embedding[:, 1]

figure, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(df["UMAP1"], df["UMAP2"], s=6, alpha=0.5, color="grey")
axes[0].set_title("the chemical space of the BBB dataset")

for value, colour, label in [(1, "#d1495b", "crosses"), (0, "#4477aa", "does not cross")]:
    subset = df[df["p_np"] == value]
    axes[1].scatter(subset["UMAP1"], subset["UMAP2"], s=6, alpha=0.6, color=colour, label=label)
axes[1].legend(markerscale=2)
axes[1].set_title("coloured by BBB penetration")

for ax in axes:
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

### Exercise 6 — Read the map

1. Are the two classes in separate regions, or mixed?
2. Find the **islands** — tight, well-separated clumps. What does an island mean chemically?
3. Connect this to Exercise 1: what does the island structure say about a *random* split?

<details>
<summary>🔍 Discussion</summary>

1. **Partly** separated: clearly red regions, clearly blue regions, and a large mixed zone — the
   geometric version of what the metrics said. A real but imperfect signal.
2. Each island is a **congeneric series**, many analogues of one scaffold. Public datasets are
   assembled from published papers, and papers report series, so the data arrives in clumps.
3. That is why the random split flattered the model: taking 20 % at random takes a slice out of
   *every* island and leaves the rest in training, so the model is asked about molecules whose
   neighbours it has memorised. The scaffold split holds out whole islands instead.

</details>

## The same map for your own target

Now with a number instead of a class, and your own compounds marked. This is the second figure
worth presenting: it shows which region of your target's chemistry the potent compounds live in,
and whether your drug sits in a crowded neighbourhood or on its own.

> ⏳ **Start the next cell now** — UMAP again, on your own target this time.

In [ ]:
# embed the library and your own compounds together, so they share one map
together = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="jaccard", random_state=42
                     ).fit_transform(np.vstack([bits[TARGET_A], probe_matrix]))
coordinates, probe_coordinates = together[:len(frame_a)], together[len(frame_a):]

figure, ax = plt.subplots(figsize=(9, 7.5))
points = ax.scatter(coordinates[:, 0], coordinates[:, 1], c=frame_a["pchembl"],
                    s=14, alpha=0.75, cmap="viridis")
figure.colorbar(points, ax=ax, label="measured pChEMBL")

for (name, _), (x, y) in zip(probe_mols.items(), probe_coordinates):
    is_drug = name == MY_DRUG
    ax.scatter(x, y, s=400 if is_drug else 170, marker="*" if is_drug else "o",
               color="#d1495b", edgecolor="white", linewidth=1.6, zorder=3)
    ax.annotate(f"  {name}", (x, y), fontsize=11, va="center", zorder=4,
                fontweight="bold" if is_drug else "normal")

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(f"{len(frame_a)} molecules measured against {TARGET_A}")
plt.tight_layout()
plt.show()

### Exercise 7 — Read your own map

1. **Is your drug inside a cluster or at the edge?** Inside means the model has analogues to learn
   from, and the applicability-domain plot will have agreed. At the edge means it does not.
2. **Where are the bright points?** If high affinity is concentrated in one or two regions, those
   regions are the series medicinal chemists have optimised, and the model's "knowledge" is mostly
   about them.
3. **Where did your shortlist land?** A shortlist clustered on top of your drug tells you nothing
   new. The useful shortlist has one compound next to your drug, one in a different bright region,
   and one somewhere dark — which is exactly the spread Exercise 7 of the pharmacophore notebook
   asked for, now visible.

<details>
<summary>🔍 One trap worth naming</summary>

UMAP distances are not quantitative. Two clusters sitting next to each other are not necessarily
more similar than two far apart — the algorithm preserves local neighbourhoods, not global
geometry. Use the map to see *structure*, and the Tanimoto numbers to make *claims*.

</details>

# Taking this to your project

| from this notebook | use it for |
|---|---|
| `fetch_target` | the published affinity data for your protein, in one line |
| the two-protein plot | your selectivity or resistance figure |
| `fit_target(..., without=my_drug)` | finding out how much of it was memorised |
| `nearest_training_similarity` | an applicability domain for every number you quote |
| scaffold split + baselines | any model you build, and any model you cite |
| the UMAP of your target | where your compounds sit, and whether the shortlist is spread |

**The one-sentence summary of Part 2.** A QSAR model interpolates within the chemistry it was
trained on. It can rank your analogues; it cannot tell you why a mutation matters, because it has
never seen a protein — only lists of substructures and numbers that people measured.

That is the gap the next notebook closes: the structure of the pocket itself.

---

An optional appendix, **`chemoinformatics_7_appendix_nlp.ipynb`**, asks a language model to pull
facts out of an abstract and finds out how far that can be trusted. Useful for the literature part
of your project; not needed for the figures above.